<a href="https://colab.research.google.com/github/pranuk050-pixel/Pyspark_Programming/blob/main/09_09_26_q3_complexdata_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('window_functions').getOrCreate()

In [ ]:

data = [(1, ['sales', 'finance', 'HR']), (2, ['sales', 'HR']), (3, ['sales', 'security'])]
df = spark.createDataFrame(data, ['id', 'dept'])
df.show()
df.printSchema()

# print for howmany depts each emp is working
# to process array column, explode()
df1= df.withColumn('dept', explode('dept'))
df1.show()
df1.printSchema()

# to generate array column ==> collect_list()
df2= df1.groupBy("id").agg(collect_list('dept').alias('dept'))
df2.show()
df2.printSchema()

+---+--------------------+
| id|                dept|
+---+--------------------+
|  1|[sales, finance, HR]|
|  2|         [sales, HR]|
|  3|   [sales, security]|
+---+--------------------+

root
 |-- id: long (nullable = true)
 |-- dept: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---+--------+
| id|    dept|
+---+--------+
|  1|   sales|
|  1| finance|
|  1|      HR|
|  2|   sales|
|  2|      HR|
|  3|   sales|
|  3|security|
+---+--------+

root
 |-- id: long (nullable = true)
 |-- dept: string (nullable = true)

+---+--------------------+
| id|                dept|
+---+--------------------+
|  1|[sales, finance, HR]|
|  3|   [sales, security]|
|  2|         [sales, HR]|
+---+--------------------+

root
 |-- id: long (nullable = true)
 |-- dept: array (nullable = false)
 |    |-- element: string (containsNull = false)



In [ ]:

json_df = spark.read.json('details.json')
json_df.show()

+---------+------+
|     city|  name|
+---------+------+
|Bangalore|charan|
+---------+------+



In [ ]:
df1= spark.read.json('devices.json')
df1.show()

+---------+--------------------+--------+---+----+------+----+-------------------+-------+
|device_id|         device_name|humidity|lat|long| scale|temp|          timestamp|zipcode|
+---------+--------------------+--------+---+----+------+----+-------------------+-------+
|        1|sensor-mac$$$ %-a...|      80| 81|  57|Celius|  30|1.447975123509765E9|  95353|
|        2|sensor-mac-able9b...|      36| 13|  56|Celius|  14|1.447975124005187E9|  96484|
|        3|sensor-mac-aboutR...|      30| 35|  54|Celius|  17|1.447975124054221E9|  96402|
|        4|sensor-mac-across...|      35| 36|  40|Celius|   6|1.447975124102157E9|  96025|
|        5|sensor-mac-afterE...|      62| 90|  91|Celius|  23|1.447975124150419E9|  95638|
|        6|sensor-mac-allBMO...|      97| 76|  77|Celius|   8|1.447975124197694E9|  95478|
|        7|sensor-mac-almost...|      30| 95|  76|Celius|   5|1.447975124246103E9|  95499|
|        8|sensor-mac-alsosT...|      74| 95|  79|Celius|  25|1.447975124291892E9|  94857|

In [ ]:
donut_df= spark.read.json('donut.json', multiLine= True)

donut_df.printSchema()
donut_df.show()
donut_df.withColumn('image_height', col('image.height'))\
        .withColumn('image_url', col('image.url')) \
        .withColumn('image_width', col('image.width')) \
        .withColumn('thumbnail_height', col('thumbnail.height'))\
        .withColumn('thumbnail_url', col('thumbnail.url')) \
        .withColumn('thumbnail_width', col('thumbnail.width')) \
        .drop('image', 'thumbnail').show()

root
 |-- id: string (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- height: long (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- width: long (nullable = true)
 |-- name: string (nullable = true)
 |-- thumbnail: struct (nullable = true)
 |    |-- height: long (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- width: long (nullable = true)
 |-- type: string (nullable = true)

+----+--------------------+----+--------------------+-----+
|  id|               image|name|           thumbnail| type|
+----+--------------------+----+--------------------+-----+
|0001|{200, images/0001...|Cake|{32, images/thumb...|donut|
+----+--------------------+----+--------------------+-----+

+----+----+-----+------------+---------------+-----------+----------------+--------------------+---------------+
|  id|name| type|image_height|      image_url|image_width|thumbnail_height|       thumbnail_url|thumbnail_width|
+----+----+-----+------------+-----------

In [ ]:
donut_df.selectExpr('id', 'image.*', 'name', 'thumbnail.*','type').show()

+----+------+---------------+-----+----+------+--------------------+-----+-----+
|  id|height|            url|width|name|height|                 url|width| type|
+----+------+---------------+-----+----+------+--------------------+-----+-----+
|0001|   200|images/0001.jpg|  200|Cake|    32|images/thumbnails...|   32|donut|
+----+------+---------------+-----+----+------+--------------------+-----+-----+



In [ ]:
print(donut_df.columns)
print(donut_df.schema)

['id', 'image', 'name', 'thumbnail', 'type']
StructType([StructField('id', StringType(), True), StructField('image', StructType([StructField('height', LongType(), True), StructField('url', StringType(), True), StructField('width', LongType(), True)]), True), StructField('name', StringType(), True), StructField('thumbnail', StructType([StructField('height', LongType(), True), StructField('url', StringType(), True), StructField('width', LongType(), True)]), True), StructField('type', StringType(), True)])
